# Day 2 — ILT 1: CDC Concepts — WAL and Logical Replication from Supabase
**Time:** Day 2, Morning
**What comes next:** ILT 2 — Lakeflow Connect & Storage Credentials, then HOL 1 — hands-on Lakeflow Connect setup

---
### What we cover today
1. What is CDC and why we need it (not just incremental load)
2. How PostgreSQL WAL works — the source of all CDC
3. Logical replication in Supabase — enabling it
4. Connecting Databricks to Supabase via JDBC (to see the mechanics)
5. Reading tables from Supabase directly into Spark

> **Instructor note:** 60 minutes. ~20 min on CDC theory (draw on board: WAL → replication slot → Lakeflow Connect), ~20 min JDBC connection live demo (to show what's happening under the hood), ~15 min WAL/logical replication setup in Supabase UI, ~5 min wrap-up. The production ingestion tool for GlobalMart is **Lakeflow Connect** (ILT 2) — this session builds the WAL/CDC mental model that Lakeflow Connect automates for you.

## Section 1 — Why CDC? Isn't Incremental Load Enough?

Recall from Day 1: **Incremental Load** reads only new rows using a watermark (date filter).  
It works for **inserts only**. But what about **updates and deletes**?

### The Problem with Incremental Load for Updates

```
customers table in Supabase:

Day 1:  CustomerID=C001, email=raj@old.com,   city=Mumbai    ← loaded to Bronze
Day 5:  CustomerID=C001, email=raj@new.com,   city=Delhi     ← customer UPDATED

Incremental Load with date filter:
  filter: WHERE created_at > last_run
  → C001 was created before last_run, so it is SKIPPED
  → Bronze still has raj@old.com and Mumbai — WRONG!
```

**CDC captures the UPDATE** — it logs that C001 changed from Mumbai to Delhi.  
That change gets propagated to Bronze → Silver → Gold automatically.

### The 3 change types CDC captures

| Operation | Example | Result in Lakehouse |
|-----------|---------|--------------------|
| **INSERT** | New customer signs up | New row added |
| **UPDATE** | Customer changes city | Existing row updated (or history tracked) |
| **DELETE** | Customer deletes account | Row removed (or marked deleted) |

## Section 2 — PostgreSQL WAL (Write-Ahead Log)

### What is WAL?

**WAL = Write-Ahead Log** — a file where PostgreSQL records every change BEFORE it commits it to the actual table.

Think of it like a bank's transaction journal:
```
Bank transaction journal:
  10:00 — Raj transferred ₹1000 to Priya
  10:01 — Account balance updated: Raj -₹1000, Priya +₹1000
  10:02 — New customer opened account

PostgreSQL WAL:
  10:00 — INSERT into customers: {id=C001, name=Raj, city=Mumbai}
  10:01 — UPDATE customers: C001 city changed Mumbai → Delhi
  10:02 — DELETE from customers: C999 removed
```

**WAL's original purpose:** crash recovery — if Postgres crashes, it replays the WAL to restore data.  
**CDC uses WAL:** we read the WAL stream instead of querying the table, so we capture EVERY change.

### Logical Replication — making WAL readable

By default, WAL is in a binary format only Postgres understands.  
**Logical replication** decodes WAL into human-readable change events (INSERT/UPDATE/DELETE).

```
WAL (binary):      0x4F02A1B4...
After decoding:    {"op": "UPDATE", "table": "customers", "id": "C001", "city": {"old": "Mumbai", "new": "Delhi"}}
```

### How to enable Logical Replication in Supabase

Supabase makes this easy through the Dashboard:

```
Supabase Dashboard
  → Database
  → Replication
  → Enable Row Level Changes for the tables you want to track
  (This sets wal_level = logical in PostgreSQL config)
```

> **We will do this in the hands-on (2:00 PM).**  
> For now — understand that enabling this is a one-time setup in Supabase UI.

## Section 3 — CDC Architecture for GlobalMart

### Full CDC Pipeline

```
Supabase PostgreSQL
       |
   WAL stream (logical replication)
       |
       v
  [Option A: Lakeflow Connect]     ← Databricks-managed, reads WAL automatically — what GlobalMart uses (ILT 2)
  [Option B: Debezium → Kafka]     ← Production-grade, self-managed, complex to operate
  [Option C: Direct JDBC polling]  ← Manual, for learning the mechanics (this session + HOL 2)
       |
       v
  Bronze (Delta)  — raw CDC events or latest snapshot
       |
       v
  Silver — MERGE new/changed rows into clean table
       |
       v
  Gold  — aggregations always reflect latest customer data
```

### What GlobalMart Actually Uses

**Lakeflow Connect** is the production tool for `orders` and `order_items` — a managed Databricks ingestion pipeline that reads the Postgres WAL through a replication slot automatically. No JDBC code, no manual slot management, no Debezium/Kafka cluster to run. You configure a connection once (`postgresqlamazon` is GlobalMart's real one) and point a pipeline at the tables you want. **ILT 2** walks through setting this up end to end.

Today's JDBC + WAL walkthrough below is deliberately manual — it exists so you understand *what Lakeflow Connect is doing for you* before you use the managed version. **HOL 2** repeats this JDBC/WAL exercise hands-on for the same reason: seeing the raw mechanics builds the intuition that makes the managed tool trustworthy instead of a black box.

| Approach | Captures Inserts | Captures Updates | Captures Deletes | Complexity | Used for |
|----------|-----------------|-----------------|-----------------|------------|----------|
| Full Load | Yes (all) | Yes (all) | Yes (by full replace) | Low | Small reference tables |
| Incremental (`created_at`) | Yes | No | No | Low | Append-only tables |
| JDBC + `updated_at` watermark | Yes | Yes | Partial (soft delete) | Low-Medium | Teaching pattern only |
| Direct JDBC + WAL replication slot | Yes | Yes | Yes | Medium | Teaching pattern (HOL 2) |
| **Lakeflow Connect (managed)** | **Yes** | **Yes** | **Yes** | **Low (managed)** | **GlobalMart production — `orders`, `order_items` (ILT 2 / HOL 1)** |
| Debezium + Kafka (self-managed) | Yes | Yes | Yes | High | Not used in this bootcamp |

## Setup — ADLS + Supabase Credentials

In [ ]:
# ─── Supabase (PostgreSQL) Credentials ───────────────────────────────────────
# WARNING: Do NOT commit this notebook to GitHub with the real password filled in
# In production, use Databricks Secrets: dbutils.secrets.get(scope, key)
# (This session teaches the raw JDBC mechanics by hand. ILT 2 replaces this
#  entirely with a governed Unity Catalog Connection + Lakeflow Connect
#  pipeline — no JDBC code, no password in a notebook cell.)

SUPABASE_HOST     = "aws-0-ap-south-1.pooler.supabase.com"
SUPABASE_PORT     = "5432"
SUPABASE_DB       = "postgres"
SUPABASE_USER     = "postgres.isqcnhvlfnjszllicxqi"
SUPABASE_PASSWORD = "YOUR_SUPABASE_PASSWORD"  # ← paste from manager's credentials

# JDBC URL — standard PostgreSQL format
jdbc_url = f"jdbc:postgresql://{SUPABASE_HOST}:{SUPABASE_PORT}/{SUPABASE_DB}"

# Connection properties — always include ssl=true for Supabase
connection_properties = {
    "user"     : SUPABASE_USER,
    "password" : SUPABASE_PASSWORD,
    "driver"   : "org.postgresql.Driver",
    "ssl"      : "true",
    "sslmode"  : "require"
}

# ─── Bronze destination — a Unity Catalog table, not a raw ADLS path ─────────
# GlobalMart's real Bronze/Silver/Gold pipeline lives in the `gbmart` catalog.
# We only need the catalog + schema name here — no storage account, container,
# or key. Unity Catalog manages where the data physically lands.
GBMART_CATALOG = "gbmart"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GBMART_CATALOG}.bronze")

print("JDBC URL:", jdbc_url)
print("Bronze target catalog.schema:", f"{GBMART_CATALOG}.bronze")
print("Credentials set — ready to connect to Supabase")

## Section 4 — Reading from Supabase via JDBC

In [ ]:
# ─── Read customers table directly from Supabase PostgreSQL ──────────────────
# This is a DIRECT database read — no CSV file needed!
# Spark connects to Supabase, runs a SELECT, and returns a DataFrame

# NOTE: The PostgreSQL JDBC driver must be installed on your Databricks cluster
# Go to: Cluster → Libraries → Install New → Maven → search 'postgresql'
# Package: org.postgresql:postgresql:42.6.0

try:
    customers_df = spark.read \
        .jdbc(
            url        = jdbc_url,
            table      = "customers",   # table name in Supabase
            properties = connection_properties
        )

    print(f"Rows read from Supabase customers: {customers_df.count():,}")
    print("\nSchema from Supabase:")
    customers_df.printSchema()
    customers_df.show(5, truncate=True)

except Exception as e:
    print(f"Connection failed: {e}")
    print("\nTroubleshooting steps:")
    print("  1. Make sure the PostgreSQL JDBC driver is installed on the cluster")
    print("  2. Check that the password is correct")
    print("  3. Try: Cluster → Libraries → Install: org.postgresql:postgresql:42.6.0")

In [ ]:
# ─── Read with a SQL query instead of full table ──────────────────────────────
# You can push SQL down to Postgres — only read what you need
# Wrap the query in parentheses and alias it as 'query'

# This reads only customers updated in the last 30 days
# This is the CDC/incremental pattern — read only changed rows
query = """
    (SELECT *
     FROM customers
     WHERE updated_at > NOW() - INTERVAL '30 days'
     ORDER BY updated_at DESC) AS recent_customers
"""

try:
    recent_customers_df = spark.read \
        .jdbc(
            url        = jdbc_url,
            table      = query,
            properties = connection_properties
        )

    print(f"Customers updated in last 30 days: {recent_customers_df.count():,}")
    recent_customers_df.show(5, truncate=True)

except Exception as e:
    print(f"Query failed: {e}")

In [ ]:
# ─── Simulate CDC: read all 10 tables from Supabase and land in Bronze ────────
# In production, this is what an ingestion pipeline does under the hood:
#   1. Connect to Supabase via JDBC
#   2. Read each table (full load or watermark filter)
#   3. Save to Bronze as a governed Unity Catalog managed table
#      (gbmart.bronze.<table> — the same destination the real Lakeflow
#      Connect pipeline in ILT 2 lands into; we do it by hand here so the
#      mechanics aren't a black box)

tables_to_ingest = [
    "customers",
    "orders",
    "orders_items",
    "products",
    "payments",
    "addresses",
    "returns",
    "suppliers",
    "payment_methods",
    "shipping_tier"
]

print(f"{'Table':<25} {'Rows':>10}  Status")
print("-" * 50)

for table_name in tables_to_ingest:
    try:
        df = spark.read.jdbc(
            url        = jdbc_url,
            table      = table_name,
            properties = connection_properties
        )
        row_count = df.count()

        df.write \
            .format("delta") \
            .mode("overwrite") \
            .saveAsTable(f"{GBMART_CATALOG}.bronze.{table_name}")

        print(f"{table_name:<25} {row_count:>10,}  Saved to gbmart.bronze from Supabase")

    except Exception as e:
        print(f"{table_name:<25} {'':>10}  ERROR: {str(e)[:60]}")

print("-" * 50)
print("When this works, you no longer need CSV files — data comes direct from Supabase!")

## Section 5 — Enabling Logical Replication in Supabase (prep for HOL 2)

> This is exactly what **Lakeflow Connect** does automatically for you when you attach a pipeline to a Postgres connection (ILT 2 / HOL 1). Doing it by hand here — and again in **HOL 2** — is what makes the managed tool make sense: you'll already know what a replication slot is and why it matters.

### Steps in Supabase Dashboard

```
1. Log in to Supabase → your project

2. Go to: Database → Replication

3. Under "Source" — you will see your tables listed

4. Toggle ON the tables you want CDC for:
   ✅ customers      (address + email changes)
   ✅ orders         (status changes: pending → shipped → delivered)
   ✅ addresses      (customers move)

5. This sets: wal_level = logical in PostgreSQL config
   Supabase handles the rest automatically
```

### What happens after enabling

```
Someone updates a customer's city in the app
    ↓
PostgreSQL writes the change to WAL
    ↓
Logical replication decodes it:
    {"op": "UPDATE", "table": "customers", "id": "C001",
     "city": {"old": "Mumbai", "new": "Delhi"}}
    ↓
We read this change event (manually via JDBC in HOL 2,
automatically via Lakeflow Connect in HOL 1)
    ↓
MERGE into Silver → Gold reflects the new city
```

> **In HOL 2:** You will make a change in your own Supabase project (update/insert/delete a row on `poc_orders`), then read the raw WAL change events back in Databricks via a replication slot — the exact mechanism Lakeflow Connect wraps for you.

## Recap

| Topic | Key Takeaway |
|-------|--------------|
| Why CDC | Incremental load misses UPDATEs and DELETEs — CDC captures all 3 |
| WAL | PostgreSQL's change log — the source of all CDC |
| Logical replication | Decodes WAL into readable INSERT/UPDATE/DELETE events |
| JDBC | Direct database connection — reads tables from Supabase into Spark (teaching pattern) |
| JDBC + `updated_at` | Simple CDC pattern — reads rows changed since last run (misses deletes) |
| Lakeflow Connect | GlobalMart's production CDC tool — manages the WAL/replication slot for you |
| Supabase setup | Enable logical replication in Dashboard → Database → Replication |

---

## What Comes Next

| Session | Topic |
|---------|-------|
| **ILT 2 (next)** | Lakeflow Connect — configuring a managed CDC ingestion pipeline + why/how to create Storage Credentials & External Locations in Unity Catalog |
| **HOL 1** | Hands-on — create your own Storage Credential + External Location, and set up a Lakeflow Connect pipeline against your own Supabase `orders`/`order_items` |
| **HOL 2** | Hands-on — the WAL/replication-slot mechanics directly via JDBC (what Lakeflow Connect does under the hood) |
| **ILT 3 / HOL 3** | Code Versioning — Databricks Repos + GitHub |

---

**INSTRUCTOR NOTE:**
Closing check:
1. *'Why doesn't a watermark filter catch deletes?'* (The row is gone — there's no `updated_at` to filter on anymore.)
2. *'What does a PostgreSQL replication slot do?'* (Bookmarks the WAL position so you can resume reading changes without gaps or duplicates.)
3. *'What is Lakeflow Connect, in one sentence?'* (A managed Databricks pipeline that reads a Postgres WAL via a replication slot and lands CDC events in Bronze — no manual JDBC/slot code required.)